# 5.a Transfer Learning with VGG16 on Caltech-101

**Section 5: Transfer learning**

Implement Object detection using Transfer Learning with CNN for the image
classification-caltec-101 dataset and VGG / ResNet pre-trained models.
Steps:
a. Set up the environment
b. Loading and preprocessing data
c. Choose the pre-trained Model (Transfer Learning)

d. Freeze the base layers and add custom layers for fine tuning
e. Object detection training
f. Compute loss
g. Evaluate the Model
h. Visualize the prediction

i. Fine Tune the Model: Fine-tuning involves unfreezing the top layers of the pre-trained
model and retraining it on the new dataset to adapt it better to your task.

**Note.** `dataset_path` is the original Windows path from the source page and is left unchanged. Point it at a local Caltech-101 image folder before running.


In [ ]:
# 1. Import necessary packages

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16


In [ ]:
# 2. Load and preprocess dataset

dataset_path = r"C:\Users\Prajwal\PICT\TE SEM 7\LP3\LP IV Updated Datasets\Object Detection\caltech-101"

train_data = keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

test_data = keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

# Get number of classes

class_names = train_data.class_names
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of Classes:", num_classes)


In [ ]:
# 3. Choose pre-trained VGG16 model

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)


In [ ]:
# 4. Freeze base model layers

base_model.trainable = False


In [ ]:
# 5. Define model architecture

model = keras.Sequential([

    layers.Input(shape=(224, 224, 3)),

    layers.Lambda(
        keras.applications.vgg16.preprocess_input
    ),

    base_model,

    layers.Flatten(),

    layers.Dense(256, activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')

])


In [ ]:
# 6. Compile model

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# 7. Train model

history = model.fit(
    train_data,
    epochs=5,
    validation_data=test_data
)


In [ ]:
# 8. Evaluate model

loss, accuracy = model.evaluate(test_data)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)


In [ ]:
# 9. Fine Tune the model

base_model.trainable = True

# Freeze all layers except last 4 layers

for layer in base_model.layers[:-4]:
    layer.trainable = False


# Compile again with low learning rate

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# Fine tune model

history_fine = model.fit(
    train_data,
    epochs=3,
    validation_data=test_data
)


In [ ]:
# 10. Evaluate fine tuned model

loss, accuracy = model.evaluate(test_data)

print("Fine Tuned Test Loss:", loss)
print("Fine Tuned Test Accuracy:", accuracy)


In [ ]:
# 11. Visualize predictions

for images, labels in test_data.take(1):

    predictions = model.predict(images)

    plt.figure(figsize=(10, 10))

    for i in range(9):

        plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy().astype("uint8"))

        predicted_class = class_names[
            np.argmax(predictions[i])
        ]

        actual_class = class_names[labels[i].numpy()]

        plt.title(
            "Predicted: " + predicted_class +
            "\nActual: " + actual_class
        )

        plt.axis('off')

    plt.show()


In [ ]:
# 12. Plot training loss and accuracy

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()


plt.subplot(1, 2, 2)

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title('Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()


Understanding the code for viva

1. What is Transfer Learning?

Transfer Learning uses a model already trained on a large dataset.

We reuse its learned features for a new dataset.

Here, we use VGG16 trained on ImageNet.

2. What is VGG16?

VGG16 is a pretrained CNN model.

It contains 16 weight layers.

It extracts features such as edges, shapes and objects.

3. What is weights='imagenet'?

Loads pretrained weights from ImageNet.

We don't need to train the entire CNN from scratch.

4. What is include_top=False?

Removes the original classification layers.

Allows us to add our own classification layers.

5. Why freeze base layers?

Prevents pretrained weights from changing initially.

Saves training time.

Preserves previously learned features.

6. What is Fine-Tuning?

Unfreezes some pretrained layers.

Retrains them using our dataset.

Helps the model adapt to new images.

7. Why use Softmax?

Produces probabilities for all object classes.

The class with the highest probability is selected.

8. Why use sparse_categorical_crossentropy?

Used for multi-class classification with integer labels.
